# Qwen3.8 × ARC: paired 50-task comparison

This notebook first scores the deterministic 50-task subset of the recorded 200-task Colab baseline, then reports available local Qwen3.8 4-bit MLX results separately. It runs locally because scoring reads ARC training solutions; it does not send answers to a model. Missing or incomplete arms are not zero scores.

Set the paths below. The Colab baseline aggregate must be the exact archived run. Scores are exploratory, not official ARC-AGI-2 results.


In [ ]:
from pathlib import Path
import hashlib, json, sys
repo = Path.cwd()
while not (repo / 'src/arc/qwen38_rsi_comparison.py').exists():
    if repo == repo.parent: raise FileNotFoundError('Run from inside the repository')
    repo = repo.parent
sys.path.insert(0, str(repo))
BASELINE = repo / 'artifacts/qwen38-quick-20261108/qwen38-quick-phasea-v1-fp8-hopper/phase_a_results.json'
CHALLENGES = repo / 'data/raw/arc-agi_training_challenges.json'
SOLUTIONS = repo / 'data/raw/arc-agi_training_solutions.json'


In [ ]:
if not all(p.is_file() for p in (BASELINE, CHALLENGES, SOLUTIONS)):
    print('Optional local Colab archive or ARC data not found; see the recorded results below.')
else:
    from src.arc.qwen38_rsi_comparison import allocate, summarize
    from src.arc.data import load_arc
    expected_baseline_sha = 'f07534b2a20704cda4eb19879b43d047bfb34d865e94e24028be552ace58f77e'
    raw = BASELINE.read_bytes()
    if hashlib.sha256(raw).hexdigest() != expected_baseline_sha:
        raise ValueError('Baseline archive does not match the recorded 200-task result.')
    record = json.loads(raw)
    proposal = json.loads((repo / 'configs/lt.phase-a.comparison-proposal.json').read_text())
    challenge_raw = CHALLENGES.read_bytes()
    if hashlib.sha256(challenge_raw).hexdigest() != proposal['dataset_sha256']:
        raise ValueError('Challenge dataset differs from Phase A.')
    tasks = json.loads(challenge_raw)
    allocation = allocate(proposal, tasks)
    if record['manifest']['phase_a_task_ids'] != proposal['task_ids']:
        raise ValueError('Baseline task IDs differ from the proposal.')
    selected = set(allocation['evaluation'])
    rows = [row for row in record['task_results'].values() if row['task_id'] in selected]
    _, verifier = load_arc(CHALLENGES, SOLUTIONS)
    scored = summarize(allocation['evaluation'], rows, verifier)
    print('Allocation SHA:', allocation['sha256'])
    print('Baseline tasks:', len(scored), '/ 50')
    print('First candidate:', sum(x['pass1'] for x in scored.values()), '/', len(scored))
    print('Either recorded candidate:', sum(x['pass2'] for x in scored.values()), '/', len(scored))
    print('Valid candidates:', sum(x['valid'] for x in scored.values()), '/', 2 * len(scored))
    print('Historical Colab baseline only; current local-arm status is below.')


The source model, user-edited Colab notebook and format recovery differ from the original strict Phase A protocol. The 50 tasks were selected after the 200-task run and are not held out. Two recorded candidates are not necessarily independent samples. Summed task runtime excludes loading and idle time.


## Local baseline snapshot — 2026-11-09

Qwen3.8-27B 4-bit MLX completed the 50-task local baseline: **2/50 first candidate**, **2/50 either recorded candidate**, and **67/100 valid candidates**. Three of the 50 raw candidate pairs were duplicates. Source: `artifacts/qwen38-local-20261109`, manifest SHA-256 `cff354ce06e7e1604725f87f1567b34f0d1408e57e2b73c1d3889e2d9fe2f626`. The selection rule retained the original prompt after its proposals scored lower on the 10 selection tasks. The unchanged-prompt repeat also completed 50 tasks: 2/50 first and either candidate, 67/100 valid candidates, with zero paired gains or losses. This repeat is not a prompt improvement. These tasks were previously evaluated and this is an exploratory, non-official result.


**Agent evolution:** all 50 tasks completed: first candidate **0/50**, either recorded candidate **2/50**, structurally valid outputs **95/100**, with eight duplicated raw pairs. Paired against baseline: one gain, one loss. Summed task inference time: 6,622.941 seconds. Selection used separate proxy tasks (3/10 versus baseline 2/10); the 50-task result is exploratory and does not establish improvement. The weight arm was not trained or scored: six extra generation passes produced 0 verified training examples and 0 verified validation examples, so the registered data gate failed (`insufficient_verified_data`).


## Local Qwen3.8 comparison

Run this cell locally after inference has created a registration. Missing arms are reported as not run. The scorer opens solutions locally and does not send them to the model.


In [ ]:
from src.cli.qwen38_local_score import report
local_run = repo / 'artifacts/qwen38-local-20261109'
if (local_run / 'manifest.json').is_file():
    local_manifest = json.loads((local_run / 'manifest.json').read_text())
    print('Provenance:', json.dumps({k: local_manifest[k] for k in ('sha256', 'model_id', 'model_revision', 'packages', 'temperature', 'max_tokens', 'comparison_previously_evaluated')}, indent=2))
    print('Allocation SHA:', local_manifest['allocation']['sha256'])
    print(json.dumps(report(repo, local_run), indent=2))
else:
    print('Local run not registered yet; no new scores are available.')
